# Week 1 Lab — The front end of a language model

**Use case.** Before a transformer can do anything, raw text has to become numbers and those
numbers have to be able to *look at each other*. You'll build the three pieces that do this,
object-oriented, and a test scoreboard tells you when each one is right.

- `BPETokenizer` — learn a subword vocabulary from a corpus, encode/decode text.
- `EmbeddingTable` — map token ids to dense vectors; find nearest neighbours.
- `SelfAttention` — scaled dot-product attention with an optional causal mask.

## How to work through this

1. Run every cell top to bottom. The **TEST SCOREBOARD** at the bottom will show all `TODO`.
2. Implement one method (find the `raise NotImplementedError`), re-run its class cell **and**
   the scoreboard cell.
3. Repeat until `ALL GREEN`. Peek at `solution.ipynb` only when stuck.

Covers: Day 01 (tokenization, embeddings), Day 02 (attention).

In [1]:
# ---- GIVEN: you don't write this ----
import numpy as np
from collections import Counter

CORPUS = (
    "the cat sat on the mat . the cat saw the dog . a dog sat on a log . "
    "the dog and the cat ran . cats and dogs run and run ."
) * 6

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(0)
print("corpus chars:", len(CORPUS))

corpus chars: 726


## 1 — `BPETokenizer`

Start from characters, repeatedly merge the most frequent adjacent pair into a new symbol until you hit `vocab_size`.

In [2]:
class BPETokenizer:
    """Byte-pair-encoding tokenizer (word-internal; whitespace splits words)."""

    def __init__(self):
        self.vocab = []          # list[str], index == token id
        self.merges = []         # list[tuple[str, str]] in learned order
        self._stoi = {}

    def _words(self, text):
        return [list(w) + ["</w>"] for w in text.split()]

    def train(self, text, vocab_size=60):
        # TODO: build self.vocab (all chars + "</w>" first), then repeatedly find the most
        #       frequent adjacent symbol pair across all words, record the merge, and apply it,
        #       until len(self.vocab) >= vocab_size or no pair repeats. Then set self._stoi.
        words = self._words(text)
        base = sorted({s for w in words for s in w})
        self.vocab, self.merges = list(base), []
        while len(self.vocab) < vocab_size:
            pairs = Counter()
            for w in words:
                for a, b in zip(w, w[1:]):
                    pairs[(a, b)] += 1
            if not pairs or pairs.most_common(1)[0][1] < 2:
                break
            (a, b), _ = pairs.most_common(1)[0]
            merged = a + b
            self.merges.append((a, b))
            self.vocab.append(merged)
            new_words = []
            for w in words:
                nw, i = [], 0
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        nw.append(merged); i += 2
                    else:
                        nw.append(w[i]); i += 1
                new_words.append(nw)
            words = new_words
        self._stoi = {s: i for i, s in enumerate(self.vocab)}
        return self

    def encode(self, text):
        # TODO: for each word, start from chars + "</w>", apply self.merges in order,
        #       then map symbols -> ids via self._stoi. Return a flat list[int].
        out = []
        for w in self._words(text):
            for a, b in self.merges:
                nw, i = [], 0
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        nw.append(a + b); i += 2
                    else:
                        nw.append(w[i]); i += 1
                w = nw
            out.extend(self._stoi[s] for s in w if s in self._stoi)
        return out

    def decode(self, ids):
        # TODO: ids -> symbols -> text. "</w>" marks a word boundary (becomes a space).
        syms = "".join(self.vocab[i] for i in ids)
        return " ".join(p for p in syms.split("</w>") if p)

    @property
    def vocab_size(self):
        return len(self.vocab)

## 2 — `EmbeddingTable`

In [3]:
class EmbeddingTable:
    """A learnable lookup table: token id -> dense vector."""

    def __init__(self, vocab_size, dim, seed=0):
        self.dim = dim
        # TODO: init self.W as a (vocab_size, dim) array, small random values
        self.W = np.random.default_rng(seed).standard_normal((vocab_size, dim)) * 0.1

    def lookup(self, ids):
        # TODO: return the rows of self.W for `ids` -> shape (len(ids), dim)
        return self.W[np.asarray(ids)]

    def nearest(self, token_id, k=3):
        # TODO: cosine similarity of self.W[token_id] to every row; return the k closest
        #       OTHER token ids, most similar first.
        v = self.W[token_id]
        sims = (self.W @ v) / (np.linalg.norm(self.W, axis=1) * np.linalg.norm(v) + 1e-9)
        order = np.argsort(-sims)
        return [int(i) for i in order if i != token_id][:k]


## 3 — `SelfAttention`

`attn(X) = softmax(QKᵀ / √d) V`, with an optional causal mask so position *t* can only attend to positions ≤ *t*.

In [4]:
class SelfAttention:
    """Single-head scaled dot-product self-attention (no learned projections for simplicity:
    Q = K = V = X, so it's a pure similarity-mixing layer)."""

    def __init__(self, causal=False):
        self.causal = causal
        self.attention_weights = None      # store the (T, T) softmax matrix from the last forward

    def forward(self, X):
        # X: (T, d).
        #   1. scores = X @ X.T / sqrt(d)
        #   2. if self.causal, set scores[i, j] = -inf for j > i
        #   3. weights = softmax(scores, axis=-1)  ; save to self.attention_weights
        #   4. return weights @ X   -> shape (T, d)
        # TODO: implement scaled dot-product self-attention with the optional causal mask
        T, d = X.shape
        scores = X @ X.T / np.sqrt(d)
        if self.causal:
            mask = np.triu(np.ones((T, T), bool), k=1)
            scores = np.where(mask, -np.inf, scores)
        w = softmax(scores, axis=-1)
        self.attention_weights = w
        return w @ X


## Scoreboard

Implement, re-run the class cell, re-run this.

In [5]:
# ============================ TEST SCOREBOARD ============================
# Run this cell any time. Each check is independent:
#   TODO  = you haven't implemented the piece it needs yet
#   FAIL  = implemented, but the result is wrong  (read the message)
#   PASS  = done
def run_checks(checks):
    width = max(len(name) for name, _ in checks)
    n_pass = n_fail = n_todo = 0
    for name, fn in checks:
        try:
            fn()
            print(f"  PASS  {name:<{width}}"); n_pass += 1
        except NotImplementedError as e:
            print(f"  TODO  {name:<{width}}   ({e})"); n_todo += 1
        except AssertionError as e:
            print(f"  FAIL  {name:<{width}}   {e}"); n_fail += 1
        except Exception as e:
            print(f"  ERR   {name:<{width}}   {type(e).__name__}: {e}"); n_fail += 1
    print(f"\n  {n_pass} pass  ·  {n_fail} fail  ·  {n_todo} todo   "
          f"({'ALL GREEN 🎉' if n_pass == len(checks) else 'keep going'})")

tok = BPETokenizer()
emb_dim = 8

def _t_train():
    tok.train(CORPUS, vocab_size=50)
    assert 30 <= tok.vocab_size <= 50, f"vocab_size={tok.vocab_size}, expected ~30-50"
    assert len(tok.merges) > 0, "no merges learned"

def _t_roundtrip():
    tok.train(CORPUS, vocab_size=50)
    txt = "the cat sat on the dog"
    assert tok.decode(tok.encode(txt)) == txt, f"round-trip failed: {tok.decode(tok.encode(txt))!r}"

def _t_bpe_compresses():
    tok.train(CORPUS, vocab_size=50)
    txt = "cats and dogs run"
    n_bpe = len(tok.encode(txt))
    n_char = len(txt.replace(" ", "")) + txt.count(" ") + 1
    assert n_bpe < n_char, f"BPE ({n_bpe}) not shorter than char-level ({n_char})"

def _t_emb_shape():
    tok.train(CORPUS, vocab_size=50)
    e = EmbeddingTable(tok.vocab_size, emb_dim)
    ids = tok.encode("the dog")
    assert e.lookup(ids).shape == (len(ids), emb_dim)

def _t_emb_nearest():
    e = EmbeddingTable(20, emb_dim)
    e.W[1] = e.W[0] + 1e-4          # make token 1 almost identical to token 0
    assert e.nearest(0, k=1)[0] == 1

def _t_attn_shape():
    X = rng.standard_normal((5, emb_dim))
    out = SelfAttention().forward(X)
    assert out.shape == X.shape

def _t_attn_rows_sum_to_1():
    X = rng.standard_normal((5, emb_dim))
    a = SelfAttention(); a.forward(X)
    assert np.allclose(a.attention_weights.sum(axis=1), 1.0), "softmax rows must sum to 1"

def _t_causal_mask():
    X = rng.standard_normal((6, emb_dim))
    a = SelfAttention(causal=True)
    out1 = a.forward(X.copy())
    X2 = X.copy(); X2[-1] += 99.0          # corrupt the LAST position
    out2 = a.forward(X2)
    assert np.allclose(out1[0], out2[0]), "position 0 changed when a LATER token changed -> mask leaks"
    assert not np.allclose(out1[-1], out2[-1]), "last position should see the change"

run_checks([
    ("tokenizer.train", _t_train),
    ("tokenizer round-trip", _t_roundtrip),
    ("BPE compresses vs chars", _t_bpe_compresses),
    ("embedding.lookup shape", _t_emb_shape),
    ("embedding.nearest", _t_emb_nearest),
    ("attention output shape", _t_attn_shape),
    ("attention weights sum to 1", _t_attn_rows_sum_to_1),
    ("causal mask blocks the future", _t_causal_mask),
])

  PASS  tokenizer.train              
  PASS  tokenizer round-trip         
  PASS  BPE compresses vs chars      
  PASS  embedding.lookup shape       
  PASS  embedding.nearest            
  PASS  attention output shape       
  PASS  attention weights sum to 1   
  PASS  causal mask blocks the future

  8 pass  ·  0 fail  ·  0 todo   (ALL GREEN 🎉)
